In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import xgboost   # must be importable to unpickle the fitted XGBRegressor

In [6]:
%pip install -q dagshub mlflow
import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)   # OAuth flow — no pasted token
import mlflow

Note: you may need to restart the kernel to use updated packages.


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=db710021-35f4-4615-bbc6-49f9031033fd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=9ac43ee77b6359a0a25c71a850d709a5c0ba85478a54578cda218ebcd80661c5




Output()

Accessing as izere23

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

In [7]:
# preprocessing module (run_pipeline builds test_full with the history the lags need)
!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline

In [8]:
# registered as "XGBoost_Walmart_final" version 1
best_pipeline = mlflow.sklearn.load_model("models:/XGBoost_Walmart_final/2")
print("loaded:", type(best_pipeline))
print(best_pipeline.named_steps.keys())

loaded: <class 'sklearn.pipeline.Pipeline'>
dict_keys(['rich_encoding', 'lags', 'yoy', 'calendar', 'preprocessing', 'model'])


In [9]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)
# patch fill_grid's IsHoliday_x/_y collision on test_full
if "IsHoliday" not in test_full.columns and "IsHoliday_x" in test_full.columns:
    test_full.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
    test_full.drop(columns=["IsHoliday_y"], inplace=True)

print("test_full:", test_full.shape)

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
test_full: (115064, 29)


In [10]:
pred = np.clip(best_pipeline.predict(test_full), 0, None)

submission = pd.DataFrame({
    "Id": (test_full["Store"].astype(str) + "_" + test_full["Dept"].astype(str)
           + "_" + test_full["Date"].dt.strftime("%Y-%m-%d")),
    "Weekly_Sales": pred,
})
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.shape)
print(submission.head())

(115064, 2)
               Id  Weekly_Sales
0  1_1_2012-11-02  38239.085938
1  1_1_2012-11-09  20842.117188
2  1_1_2012-11-16  20602.335938
3  1_1_2012-11-23  21935.595703
4  1_1_2012-11-30  26206.753906
